# Analyzing Data in Python

The sample dataset contains obfuscated GA360 data for August 1, 2017 from the [Google Merchandise Store](https://shop.googlemerchandisestore.com/), a real ecommerce store selling Google branded merchandise ([Source](https://support.google.com/analytics/answer/7586738)).

This data is typical for what you'd see for an ecommerce website. It contains session data with traffic source, location and transcation info. Other data has been anonymized. The data is available for in a publicly available Google Sheet. Python is used to analyze this dataset, getting more insights into what is driving sales on this particular day.

## 1: Import packages

In [1]:
import pandas as pd
import plotly.express as px

## 2: Import the session data

Import all data from the Google Sheet.

In [2]:
df = pd.read_excel('data/ga_sample_data.xlsx')
df.head()

,visitId,visitStartTime,visitNumber,channelGrouping,browser,operatingSystem,deviceCategory,continent,subContinent,country,transactions
0,1501591568,1501591568,1,Organic Search,Chrome,Windows,desktop,Europe,Southern Europe,Greece,NaN
1,1501589647,1501589647,2,Referral,Chrome,Windows,desktop,Asia,Southern Asia,India,NaN
2,1501616621,1501616621,1,Referral,Chrome,Windows,desktop,Europe,Northern Europe,United Kingdom,NaN
3,1501601200,1501601200,1,Referral,Firefox,Windows,desktop,Americas,Northern America,United States,NaN
4,1501615525,1501615525,1,Referral,Chrome,Windows,desktop,Americas,Northern America,United States,NaN


## 3: Clean the data

Let's inspect the types of the columns, and make adjustments where needed.

In [3]:
df.dtypes

visitId              int64
visitStartTime       int64
visitNumber          int64
channelGrouping     object
browser             object
operatingSystem     object
deviceCategory      object
continent           object
subContinent        object
country             object
transactions       float64
dtype: object

In [4]:
df_clean = df.copy()
df_clean['visitStartTime'] = pd.to_datetime(df_clean['visitStartTime'], unit='s').dt.tz_localize('UTC')
df_clean

,visitId,visitStartTime,visitNumber,channelGrouping,browser,operatingSystem,deviceCategory,continent,subContinent,country,transactions
0,1501591568,2017-08-01 12:46:08+00:00,1,Organic Search,Chrome,Windows,desktop,Europe,Southern Europe,Greece,NaN
1,1501589647,2017-08-01 12:14:07+00:00,2,Referral,Chrome,Windows,desktop,Asia,Southern Asia,India,NaN
2,1501616621,2017-08-01 19:43:41+00:00,1,Referral,Chrome,Windows,desktop,Europe,Northern Europe,United Kingdom,NaN
3,1501601200,2017-08-01 15:26:40+00:00,1,Referral,Firefox,Windows,desktop,Americas,Northern America,United States,NaN
4,1501615525,2017-08-01 19:25:25+00:00,1,Referral,Chrome,Windows,desktop,Americas,Northern America,United States,NaN
...,...,...,...,...,...,...,...,...,...,...,...
2551,1501638264,2017-08-02 01:44:24+00:00,1,Referral,Chrome,Macintosh,desktop,Americas,Northern America,United States,NaN
2552,1501605330,2017-08-01 16:35:30+00:00,1,Organic Search,Chrome,Macintosh,desktop,Americas,Northern America,United States,NaN
2553,1501615026,2017-08-01 19:17:06+00:00,3,Organic Search,Chrome,Macintosh,desktop,Americas,Northern America,United States,NaN
2554,1501627131,2017-08-01 22:38:51+00:00,16,Organic Search,Chrome,Windows,desktop,Americas,Northern America,United States,1.0


In [5]:
df_clean.dtypes

visitId                          int64
visitStartTime     datetime64[ns, UTC]
visitNumber                      int64
channelGrouping                 object
browser                         object
operatingSystem                 object
deviceCategory                  object
continent                       object
subContinent                    object
country                         object
transactions                   float64
dtype: object

## 4: Explore the data

Use the predefined plotting function to explore different grouped session counts as either a bar chart or a pie chart.

In [8]:
def plot_sessions_per_group(df, group, viz_type='bar', template="plotly_white"):
    sessions_per_group = df.groupby(group).size().reset_index(name='sessions').sort_values(by='sessions', ascending=False)
    fig = None                                                                                  
    if viz_type == 'bar':
        fig = px.bar(sessions_per_group,
                      x=group,
                      y='sessions',
                      title=f'Number of sessions per {group}',
                      text='sessions')
        fig.update_traces(textposition='outside')
        fig.update_layout(uniformtext_minsize=8, uniformtext_mode='hide')
    elif viz_type == 'pie':
        fig = px.pie(sessions_per_group,
                      names=group,
                      values='sessions',
                      hole=0.5,
                      title=f'Distribution of sessions per {group}')
        fig.update_layout(template=template)
    else:
        raise ValueError("viz_type can only be 'bar' or 'pie'")
    # Common layout styling
    fig.update_layout(title_x=0.5, title_font_size=20, title_font_family='Arial', title_font_color='darkblue')
    fig.update_layout(template=template, margin=dict(l=40, r=40, t=80, b=40))
    
    return fig

In [9]:
plot_sessions_per_group(df_clean, 'continent')

In [10]:
plot_sessions_per_group(df_clean, 'channelGrouping', viz_type='pie')

In [11]:
plot_sessions_per_group(df_clean, 'deviceCategory', viz_type='pie')

## 5: Show hourly visits per channelGrouping

Adjust the code below to show the hourly visits throughout the day per `channelGrouping`.

In [12]:
df_clean['visitStartTime'] = df_clean['visitStartTime'].dt.tz_convert('America/Los_Angeles')
df_clean['hour'] = df_clean['visitStartTime'].dt.to_period('H')
visits_per_month = df_clean.groupby(['hour', 'channelGrouping']).size().reset_index(name='visits')
visits_per_month['hour'] = visits_per_month['hour'].astype(str)
px.line(visits_per_month, x='hour', y='visits', color='channelGrouping', title='Visits per hour throughout the day', markers=True)

C:\Users\nasym\AppData\Local\Temp\ipykernel_10740\4022200030.py:2: UserWarning:

Converting to PeriodArray/Index representation will drop timezone information.

C:\Users\nasym\AppData\Local\Temp\ipykernel_10740\4022200030.py:2: FutureWarning:

'H' is deprecated and will be removed in a future version, please use 'h' instead.



## 6 : which channel leads to most transactions per session?

In [13]:
channel_grouping_stats = (
    df_clean
    .groupby('channelGrouping')
    .agg({'transactions': 'sum', 'visitId': 'size'})
    .reset_index()
)
channel_grouping_stats['transactions_per_session'] = channel_grouping_stats['transactions'] / channel_grouping_stats['visitId']
channel_grouping_stats_sorted = channel_grouping_stats.sort_values('transactions_per_session')

px.bar(channel_grouping_stats_sorted,
       x='channelGrouping',
       y='transactions_per_session',
       title='Transactions per Session by Channel Grouping')